Week 3: ML pipelines

Data source: the NYC taxi dataset

In [26]:
import pandas as pd
import numpy as np
import seaborn as sns
import pickle

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [3]:
pd.__version__, np.__version__, sns.__version__

('2.2.3', '2.2.5', '0.13.2')

In [4]:
import mlflow
mlflow.__version__

'2.22.0'

In [6]:
TRACKING_URI = "http://127.0.0.1:5000"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment('NYC-taxi-experiment')

2025/06/08 13:11:47 INFO mlflow.tracking.fluent: Experiment with name 'NYC-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1749381107794, experiment_id='1', last_update_time=1749381107794, lifecycle_stage='active', name='NYC-taxi-experiment', tags={}>

In [11]:
train_fn =  'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet'
val_fn =  'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet'

In [16]:
# embed all the preprocessing in a function
def preprocess_df(fn):
    df = pd.read_parquet(fn)
    # create duration (in minutes) feature
    df['duration'] = (
        df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    ).dt.total_seconds().div(60.)
    # filter out outliers: trips should be between 1 and 60 minutes.
    df = df[
        (1 <= df.duration) & (df.duration <= 60.)
    ]
    categorical_cols = ['PULocationID', 'DOLocationID']
    # convert categorical columns to string data type
    df[categorical_cols] = df[categorical_cols].astype(str)
    df['PU_DO'] = df[categorical_cols[0]] + '_' + df[categorical_cols[1]]
    return df

In [17]:
df_train = preprocess_df(train_fn)
df_valid = preprocess_df(val_fn)

In [18]:
df_train.shape, df_valid.shape

((73908, 22), (61921, 22))

In [19]:
categorical = ['PU_DO']
numerical   = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

valid_dicts = df_valid[categorical + numerical].to_dict(orient='records')
X_valid = dv.transform(valid_dicts)

In [22]:
target = 'duration'
y_train = df_train[target].values
y_valid = df_valid[target].values

In [20]:
import xgboost as xgb

In [23]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_valid, label=y_valid)

In [24]:
mlflow.xgboost.autolog(disable=True)
# signature = mlflow.models.infer_signature(
#     X_valid, y_valid
# )

In [28]:
best_result = {
    'learning_rate': 0.2455,
    'max_depth': 90,
    'min_child_weight': 5.1467,
    'reg_alpha': 0.3175,
    'reg_lambda': 0.2810
}

In [29]:
with mlflow.start_run():
    mlflow.log_params(best_result)
    booster = xgb.train(
        params=best_result,
        dtrain=train,
        num_boost_round=100,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50,
        verbose_eval=False,
    )
    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(
        y_valid, y_pred
    )
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.pkl", "wb") as fw:
        pickle.dump(dv, fw)
    mlflow.log_artifact("models/preprocessor.pkl", artifact_path="preprocessor")
    model_info = mlflow.xgboost.log_model(
        booster, 
        artifact_path='models_mlflow',
        # signature=signature,
    )

/Users/nabe/miniconda3/envs/mlopszoomcamp/lib/python3.13/site-packages/mlflow/xgboost/__init__.py:168: UserWarning: [13:34:51] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1745056754219/work/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  xgb_model.save_model(model_data_path)
2025/06/08 13:34:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run luxuriant-skink-645 at: http://127.0.0.1:5000/#/experiments/1/runs/508bcc23ba5c4b79bcb140c75a97c3ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
